In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS nyc_taxi.sandbox
COMMENT 'Experiments and learning, not part of the pipeline';

CREATE VOLUME IF NOT EXISTS nyc_taxi.sandbox.delta_lab
COMMENT 'Sandbox for exploring Delta Lake internals';

In [0]:
from pyspark.sql import functions as F

LAB_PATH = "/Volumes/nyc_taxi/sandbox/delta_lab/trips_demo"

sample = (
    spark.table("nyc_taxi.bronze.yellow_trips")
    .select("VendorID", "tpep_pickup_datetime", "passenger_count", "fare_amount")
    .limit(1000)
)

sample.write.format("delta").mode("overwrite").save(LAB_PATH)

In [0]:
%sql
LIST '/Volumes/nyc_taxi/sandbox/delta_lab/trips_demo/_delta_log/';

In [0]:
display(spark.read.text(LAB_PATH + "/_delta_log/00000000000000000000.json"))

In [0]:
sample.limit(100).write.format("delta").mode("append").save(LAB_PATH)
display(spark.read.text(LAB_PATH + "/_delta_log/00000000000000000001.json"))

In [0]:
%sql
DELETE FROM delta.`/Volumes/nyc_taxi/sandbox/delta_lab/trips_demo` WHERE passenger_count = 1;

In [0]:
display(spark.read.text(LAB_PATH + "/_delta_log/00000000000000000002.json"))

In [0]:
bad = sample.withColumn("tip_pct", F.lit(0.15))
bad.write.format("delta").mode("append").save(LAB_PATH)

In [0]:
bad.write.format("delta").mode("append").option("mergeSchema", "true").save(LAB_PATH)
display(spark.read.text(LAB_PATH + "/_delta_log/00000000000000000004.json"))

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/nyc_taxi/sandbox/delta_lab/trips_demo`;